In [ ]:
# 1. pytest 
# 2. cache
# 3. pydantic object and output parser
# 4. token counter(cost analysis)
# 5. history or memory 
# 6. evaluation
# 7. guadrails
# 8. adavance RAG

## as a experiment will do in the notebook
### then as a assignment you can integrate it in end to end project

## Next Saturday there will demonstration of this concept (whoever will comeplete it
# that person will give the demo)

### next week satuday 2-2:30

### then will start with the next project

#For first winner there is prize money:

Assignment on first Project:

you have to make enable this project for every document(.ppt,.docx,.md,.txt,.pdf,.xlxs,.csv,anysqldb)
you have to add a code for dealing with table and images data also
you have to add the evalation matrix using the DeepEval
write at least 10 test cases
this cases should be vaildate before the commit and after the commit
add one login screen to your protal
use the langchain inmemory cache and implement inside the project
deadline for this assignment is till 5th of september(friday)
inner will present the entire solution to the whole class (30 minute presentation would be there)
prize: money or course or giftpack(there will be only one winnner)

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from utils.model_loader import ModelLoader

In [ ]:
loader = ModelLoader()


In [ ]:
llm = loader.load_llm()
print(f"LLM Loaded: {llm}")
result = llm.invoke("Hello, how are you?")
print(f"LLM Result: {result.content}")

In [ ]:
parser = JsonOutputParser()


In [ ]:
parser.get_format_instructions()


In [ ]:
prompt = PromptTemplate.from_template(
    "Give me a JSON with keys 'title' and 'summary' for this topic: {topic}\n{format_instructions}"
).partial(format_instructions=parser.get_format_instructions())

In [ ]:
prompt

In [ ]:
chain = prompt | llm | parser


In [ ]:
result = chain.invoke({"topic": "LangChain for RAG"})


In [ ]:
result

In [ ]:
from langchain.output_parsers import OutputFixingParser


In [ ]:
json_parser = JsonOutputParser()


In [ ]:
# Fixing wrapper
fixing_parser = OutputFixingParser.from_llm(parser=json_parser, llm=llm)

In [ ]:
# Broken JSON
bad_output = """
title: LangChain RAG
summary LangChain helps with retrieval augmented generation...
"""

In [ ]:
fixed_result = fixing_parser.parse(bad_output)


In [ ]:
print(fixed_result)


# from langchain.output_parsers import OutputFixingParser
# from langchain_core.output_parsers import JsonOutputParser

# json_parser = JsonOutputParser()
# safe_parser = OutputFixingParser.from_llm(parser=json_parser, llm=llm)

# # Use safe_parser instead of json_parser
# chain = prompt | llm | safe_parser

In [ ]:
# Wrap with cache
Model_Cache = {}

import time
def cached_model(query):
    start_time = time.time()
    if Model_Cache.get(query):
        print("***CACHE HIT***")
        end_time = time.time()
        elapsed = end_time - start_time
        print(f"EXECUTION TIME: {elapsed:.2f} seconds")
        return Model_Cache.get(query)
    else:
        print("***CACHE MISS – EXECUTING MODEL***")
        start_time = time.time()
        response = llm.invoke(query)
        end_time = time.time()
        elapsed = end_time - start_time
        print(f"EXECUTION TIME: {elapsed:.2f} seconds")
        Model_Cache[query] = response
        return response

In [ ]:
query="hi"
response = cached_model(query)
print(response)

In [ ]:
query="can you give me 1000 words essay on independence?"
response = cached_model(query)
print(response)

In [ ]:
query="can you give me 1000 words essay on independence?"
response = cached_model(query)
print(response)

In [ ]:
embedding_model = loader.load_embeddings()
print(f"Embedding Model Loaded: {embedding_model}")
result = embedding_model.embed_query("Hello, how are you?")
print(f"Embedding Result: {result}")

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableMap, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
from langchain.schema import Document

documents = [
    Document(page_content="The Earth is the third planet from the Sun and the only known planet to support life. It has a diverse climate, ranging from arctic to tropical zones, and supports ecosystems across seven continents and five oceans."),

    Document(page_content="The Industrial Revolution, beginning in the 18th century, drastically transformed human societies by shifting from manual labor to machine-based manufacturing, leading to urbanization and economic expansion globally."),

    Document(page_content="The United Nations, established in 1945 after World War II, is an international organization founded to promote peace, security, human rights, and cooperation among countries. It has 193 member states."),

    Document(page_content="The global economy is an interconnected system involving trade, investment, and financial flows across countries. Major players include the United States, China, the European Union, and emerging markets like India and Brazil."),

    Document(page_content="Climate change refers to long-term shifts in temperatures and weather patterns. It is largely driven by human activities like burning fossil fuels, deforestation, and industrial emissions, leading to global warming and sea level rise."),

    Document(page_content="Democracy is a political system in which citizens exercise power by voting. Modern democracies typically have institutions for free elections, rule of law, freedom of expression, and checks and balances."),

    Document(page_content="The Internet has revolutionized communication, commerce, and education worldwide. Originating from military research in the 1960s, it now connects over 5 billion people, enabling instant global information exchange."),

    Document(page_content="Renewable energy sources like solar, wind, hydro, and geothermal are critical for a sustainable future. They offer alternatives to fossil fuels, reducing carbon emissions and reliance on finite resources."),

    Document(page_content="The World Health Organization (WHO) is a UN agency focused on global health issues. It coordinates international efforts to monitor diseases, set health standards, and respond to pandemics like COVID-19."),

    Document(page_content="Globalization is the process of increasing interaction and integration among people, companies, and governments worldwide. It has led to greater economic growth but also raised concerns about inequality and cultural homogenization.")
]

In [ ]:
# Chroma vector DB with persistent storage
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory="./chroma_db"  # Disk path for persistence
)

In [ ]:
# Optional: Persist manually (though auto-persistence happens internally)
vector_store.persist()

In [ ]:
retriever = vector_store.as_retriever()


In [ ]:
prompt = PromptTemplate.from_template(
    """
    Use the following context to answer the question.
    If you don't know the answer, just say you don't know. Don't try to make up an answer.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)

In [ ]:
# LCEL RAG Chain step-by-step
rag_chain = (
    RunnableMap({
        "context": retriever | (lambda docs: "\n\n".join([doc.page_content for doc in docs])),
        "question": RunnablePassthrough()
    })
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# Wrap with cache
RAG_Cache = {}

In [ ]:
import time
def cached_rag_chain(query):
    start_time = time.time()
    if RAG_Cache.get(query):
        print("***CACHE HIT***")
        end_time = time.time()
        elapsed = end_time - start_time
        print(f"EXECUTION TIME: {elapsed:.2f} seconds")
        return RAG_Cache.get(query)
    else:
        print("***CACHE MISS – EXECUTING MODEL***")
        start_time = time.time()
        response = llm.invoke(query)
        end_time = time.time()
        elapsed = end_time - start_time
        print(f"EXECUTION TIME: {elapsed:.2f} seconds")
        RAG_Cache[query] = response
        return response

In [ ]:
query = "what is japan economy in 2024 and relation with north korea?"
response = cached_rag_chain(query)
print(response)

In [ ]:
query = "what is langchain framework?"
response = cached_rag_chain(query)
print(response)

In [ ]:
query = "Why United Nations, established in 1945 after World War II?"
response = cached_rag_chain(query)
print(response)

#Cache Using LangChain


In [ ]:
from langchain.cache import InMemoryCache
from langchain.globals import set_llm_cache
from typing import Any, Dict, Tuple

In [ ]:
class DebuggableCache(InMemoryCache):
    def __init__(self):
        super().__init__()
        self._cache: Dict[Tuple[str, str], Any] = {}

    def lookup(self, prompt: str, llm_string: str):
        return self._cache.get((prompt, llm_string))

    def update(self, prompt: str, llm_string: str, return_val: Any):
        self._cache[(prompt, llm_string)] = return_val

    def view_cache(self):  # this is our custom method
        return self._cache

In [ ]:
dbg_cache = DebuggableCache()
set_llm_cache(dbg_cache)

In [ ]:
response = llm.invoke("What is the capital of France?")


In [ ]:
print("LLM Response:", response)


In [ ]:
print("\nCache Contents:")
for k, v in dbg_cache.view_cache().items():
    print(f"Prompt: {k[0]} | Cached Output: {v}")

In [ ]:
response = llm.invoke("What is the capital of France?")


In [ ]:
print("\nCache Contents:")
for k, v in dbg_cache.view_cache().items():
    print(f"Prompt: {k[0]} | Cached Output: {v}")

In [ ]:
response = llm.invoke("What is the capital of india?")
print("LLM Response:", response)

In [ ]:
print("\nCache Contents:")
for k, v in dbg_cache.view_cache().items():
    print(f"Prompt: {k[0]} | Cached Output: {v}")

In [ ]:
response = llm.invoke("What is the capital of india?")
print("LLM Response:", response)

In [ ]:
response = llm.invoke("give me 1000 lines of essay on science and give the importance of it regarding mathematics?")
print("LLM Response:", response)

In [ ]:
from langchain.callbacks import get_openai_callback
with get_openai_callback() as cb:
    response = llm.invoke("Tell me a joke about LangChain and ECS")
    print("Response:", response.content)
    print("Token Usage Stats:", cb)

In [ ]:
questions = [
    "What is Retrieval-Augmented Generation?",
    "How does FAISS indexing work?",
    "What is the difference between fine-tuning and RAG?"
]

with get_openai_callback() as cb:
    for q in questions:
        answer = llm.invoke(q)
        print(f"\nQ: {q}\nA: {answer.content}")

    print("\n=== Token Usage Summary ===")
    print(f"Total Tokens: {cb.total_tokens}")
    print(f"Prompt Tokens: {cb.prompt_tokens}")
    print(f"Completion Tokens: {cb.completion_tokens}")
    print(f"Total Cost (USD): ${cb.total_cost:.6f}")